# WakeRef — Fine-tuning Whisper (**passes de JIB**)

Entraîne un Whisper à **transcrire une passe de jib entière** (séquence d'atomes), pas un trick isolé. Pipeline distinct de celui des tricks.

**Avant de commencer :** menu **Exécution → Modifier le type d'exécution → T4 GPU**.

Lance les cellules **dans l'ordre** (▶). Étapes : install → dataset → script → entraînement → téléchargement.

## 1. Installer les outils (~2 min)

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate jiwer librosa soundfile

## 2. Récupérer le dataset jib (.zip) depuis Google Drive → wav
Mets ton `.zip` dataset dans Google Drive, puis renseigne `ZIP` ci-dessous. Drive ↔ Colab = transfert **Google-interne (rapide)**, contrairement à `files.upload()` (bridé). Repli « upload manuel » en commentaire.

In [ ]:
# Dataset depuis Google Drive (rapide) — plutôt que files.upload() (bridé).
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/wakeref-voix-dataset-jib.zip'   # <- adapte le nom/chemin

# Repli (upload depuis ta machine) : décommente les 2 lignes ci-dessous.
# from google.colab import files
# ZIP = '/content/' + next(iter(files.upload()))

!unzip -q -o "$ZIP" -d ds
!for f in ds/audio/*.webm; do ffmpeg -nostdin -loglevel error -y -i "$f" -ar 16000 -ac 1 "${f%.webm}.wav"; done
print('OK — dataset jib pret')

## 3. Créer le script d'entraînement
Réponse attendue : `Writing finetune_whisper_jib.py`.

In [ ]:
%%writefile finetune_whisper_jib.py
#!/usr/bin/env python
# Fine-tune Whisper sur des PASSES DE JIB entières (export .zip 'jib' de /judge/voix).
#
# != pipeline tricks : ici la cible (`transcription`) est le TEXTE COMPLET de la passe
# corrige par le juge (ex. "ollie backlip to boardslide 270 out"), PAS un nom de trick
# isole. Le modele apprend a TRANSCRIRE une sequence compositionnelle d'atomes
# (structure + rotations + slides + grabs + tricks) -> il generalise a des combinaisons
# jamais vues tant que chaque atome a ete entendu dans des contextes varies.
#
# Base = whisper-small (meilleur FR sur le jargon dense). Audio charge a la volee dans
# le collator (pas de datasets.map -> pas de saturation memoire). Pense pour Colab T4.
#
#   python finetune_whisper_jib.py --data ./ds --base openai/whisper-small --out ./whisper-wakeref-jib
#
import argparse, os
from dataclasses import dataclass
from typing import Any
import pandas as pd
import librosa
import torch
from datasets import Dataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
import evaluate

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True, help="dossier decompresse du zip (metadata.csv + audio/)")
ap.add_argument("--base", default="openai/whisper-small")
ap.add_argument("--out",  default="./whisper-wakeref-jib")
ap.add_argument("--epochs", type=float, default=10.0)
ap.add_argument("--bs", type=int, default=8)
ap.add_argument("--grad_accum", type=int, default=2)
args = ap.parse_args()

# -- Donnees : on garde (chemin, texte de passe) ; l'audio est lu dans le collator --
df = pd.read_csv(os.path.join(args.data, "metadata.csv"))
df = df[df["transcription"].notna() & (df["transcription"].astype(str).str.strip() != "")]
# Si le CSV melange tricks + jib (export global), ne garder que les passes jib.
if "kind" in df.columns and (df["kind"].astype(str) == "jib").any():
    df = df[df["kind"].astype(str) == "jib"]
df["path"] = df["file_name"].apply(lambda p: os.path.join(args.data, p))
print(f"{len(df)} passes jib")

ds = Dataset.from_pandas(df[["path", "transcription"]], preserve_index=False)
ds = ds.train_test_split(test_size=0.1, seed=42)

processor = WhisperProcessor.from_pretrained(args.base, language="french", task="transcribe")

def resolve(p):
    # Prefere un .wav 16 kHz pre-converti s'il existe (decodage webm fiable via ffmpeg).
    wav = os.path.splitext(p)[0] + ".wav"
    return wav if os.path.exists(wav) else p

# -- Collator : charge l'audio + calcule mel + tokenise, par batch --------------
@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        audios = [librosa.load(resolve(f["path"]), sr=16000, mono=True)[0] for f in features]
        feats = self.processor.feature_extractor(audios, sampling_rate=16000, return_tensors="pt")
        lab = self.processor.tokenizer([f["transcription"] for f in features], padding=True, return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        return {"input_features": feats.input_features, "labels": labels}

# -- Modele ---------------------------------------------------------------------
model = WhisperForConditionalGeneration.from_pretrained(args.base)
model.generation_config.language = "french"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

metric = evaluate.load("wer")
def compute_metrics(p):
    pred = p.predictions
    pred[pred == -100] = processor.tokenizer.pad_token_id
    labels = p.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(pred, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=ps, references=ls)}

targs = Seq2SeqTrainingArguments(
    output_dir=args.out,
    per_device_train_batch_size=args.bs,
    per_device_eval_batch_size=args.bs,
    gradient_accumulation_steps=args.grad_accum,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=args.epochs,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,     # une passe est plus longue qu'un nom de trick
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=1,            # ne garde qu'un checkpoint (sinon le dossier pese des Go)
    remove_unused_columns=False,   # le collator a besoin des colonnes path/transcription
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model, args=targs,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=Collator(processor), compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()
trainer.save_model(args.out)
processor.save_pretrained(args.out)
print(f"\n[ok] modele fine-tune jib -> {args.out}")

## 4. Entraîner (~30-50 min)
La **loss** doit descendre ; la **WER** (taux d'erreur mot) s'affiche à chaque epoch. Le meilleur modèle est gardé automatiquement.

> whisper-small est plus lourd que base : si tu as une erreur mémoire (OOM) sur T4, relance en ajoutant `--bs 4 --grad_accum 4`.

In [ ]:
!python finetune_whisper_jib.py --data ds --base openai/whisper-base --out whisper-wakeref-jib

## 5. Récupérer le modèle (zip propre)
On retire les checkpoints d'entraînement (lourds, inutiles) avant de zipper.

In [ ]:
!cd whisper-wakeref-jib && rm -rf checkpoint-* && cd .. && zip -qr whisper-wakeref-jib.zip whisper-wakeref-jib
from google.colab import files
files.download('whisper-wakeref-jib.zip')